# Model Experiments

This notebook contains experiments with different models, hyperparameter tuning, and performance evaluation.

In [ ]:
# Import necessary libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import time

# Add src directory to path
import sys
sys.path.append('../src')

# Import project modules
from config import SVM_C, SVM_KERNEL, SVM_GAMMA
from dataset import create_dataset
from features import extract_hog_features_batch, extract_color_histogram, extract_lbp_features
from train import train_svm_classifier, evaluate_model, perform_cross_validation
from evaluate import calculate_metrics, plot_confusion_matrix, generate_evaluation_report

## Load and Prepare Dataset

In [ ]:
# Create dataset
print("Loading and preparing dataset...")
X_train, X_test, y_train, y_test, label_encoder = create_dataset()

print(f"Dataset loaded:")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")
print(f"  Classes: {label_encoder.classes_}")
print(f"  Image shape: {X_train[0].shape}")

## Feature Extraction

In [ ]:
# Extract different types of features
print("Extracting features...")

# HOG features
print("  Extracting HOG features...")
X_train_hog = extract_hog_features_batch(X_train)
X_test_hog = extract_hog_features_batch(X_test)
print(f"  HOG features shape: {X_train_hog.shape}")

# Color histogram features
print("  Extracting color histogram features...")
X_train_color = np.array([extract_color_histogram((img * 255).astype(np.uint8), bins=32) 
                         for img in X_train])
X_test_color = np.array([extract_color_histogram((img * 255).astype(np.uint8), bins=32) 
                        for img in X_test])
print(f"  Color histogram features shape: {X_train_color.shape}")

# LBP features
print("  Extracting LBP features...")
X_train_lbp = np.array([extract_lbp_features((img * 255).astype(np.uint8)) 
                       for img in X_train])
X_test_lbp = np.array([extract_lbp_features((img * 255).astype(np.uint8)) 
                      for img in X_test])
print(f"  LBP features shape: {X_train_lbp.shape}")

# Combined features
X_train_combined = np.hstack([X_train_hog, X_train_color, X_train_lbp])
X_test_combined = np.hstack([X_test_hog, X_test_color, X_test_lbp])
print(f"  Combined features shape: {X_train_combined.shape}")

feature_sets = {
    'HOG': (X_train_hog, X_test_hog),
    'Color': (X_train_color, X_test_color),
    'LBP': (X_train_lbp, X_test_lbp),
    'Combined': (X_train_combined, X_test_combined)
}

## Baseline Model Comparison

In [ ]:
# Define models to test
models = {
    'SVM': SVC(C=SVM_C, kernel=SVM_KERNEL, gamma=SVM_GAMMA, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Naive Bayes': GaussianNB()
}

# Test models on different feature sets
results = {}

print("Testing baseline models...")
for feature_name, (X_tr, X_te) in feature_sets.items():
    print(f"\nTesting with {feature_name} features:")
    
    # Scale features for some models
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)
    
    feature_results = {}
    
    for model_name, model in models.items():
        print(f"  Training {model_name}...")
        
        # Use scaled features for most models except Random Forest
        if model_name == 'Random Forest':
            X_tr_use, X_te_use = X_tr, X_te
        else:
            X_tr_use, X_te_use = X_tr_scaled, X_te_scaled
        
        # Train model
        start_time = time.time()
        model.fit(X_tr_use, y_train)
        training_time = time.time() - start_time
        
        # Make predictions
        start_time = time.time()
        y_pred = model.predict(X_te_use)
        prediction_time = time.time() - start_time
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        metrics = calculate_metrics(y_test, y_pred)
        
        feature_results[model_name] = {
            'accuracy': accuracy,
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1_score': metrics['f1_score'],
            'training_time': training_time,
            'prediction_time': prediction_time,
            'model': model
        }
        
        print(f"    Accuracy: {accuracy:.4f}, Time: {training_time:.2f}s")
    
    results[feature_name] = feature_results

## Results Visualization

In [ ]:
# Create results DataFrame
results_df = []
for feature_name, feature_results in results.items():
    for model_name, metrics in feature_results.items():
        results_df.append({
            'Feature Set': feature_name,
            'Model': model_name,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1-Score': metrics['f1_score'],
            'Training Time': metrics['training_time'],
            'Prediction Time': metrics['prediction_time']
        })

results_df = pd.DataFrame(results_df)

# Display results table
print("Model Comparison Results:")
print(results_df.round(4).to_string(index=False))

# Plot accuracy comparison
plt.figure(figsize=(15, 8))
pivot_accuracy = results_df.pivot(index='Model', columns='Feature Set', values='Accuracy')
sns.heatmap(pivot_accuracy, annot=True, fmt='.3f', cmap='YlOrRd')
plt.title('Model Accuracy by Feature Set')
plt.tight_layout()
plt.show()

## Detailed Analysis of Best Models

In [ ]:
# Find best performing model
best_row = results_df.loc[results_df['Accuracy'].idxmax()]
best_feature_set = best_row['Feature Set']
best_model_name = best_row['Model']
best_accuracy = best_row['Accuracy']

print(f"Best performing model: {best_model_name} with {best_feature_set} features")
print(f"Accuracy: {best_accuracy:.4f}")

# Get the best model and feature set
best_model = results[best_feature_set][best_model_name]['model']
X_train_best, X_test_best = feature_sets[best_feature_set]

# Scale features if needed
if best_model_name != 'Random Forest':
    scaler = StandardScaler()
    X_train_best = scaler.fit_transform(X_train_best)
    X_test_best = scaler.transform(X_test_best)

# Retrain on full training set
print(f"\nRetraining best model on full training set...")
best_model.fit(X_train_best, y_train)

# Make predictions
y_pred = best_model.predict(X_test_best)

# Detailed evaluation
print("\nDetailed Evaluation:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f'Confusion Matrix - {best_model_name} ({best_feature_set})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Hyperparameter Tuning for SVM

In [ ]:
# Hyperparameter tuning for SVM with HOG features
print("Performing hyperparameter tuning for SVM...")

# Use HOG features for SVM tuning
X_train_hog_scaled = StandardScaler().fit_transform(X_train_hog)
X_test_hog_scaled = StandardScaler().fit_transform(X_test_hog)

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'degree': [2, 3, 4]  # Only used for poly kernel
}

# Create SVM classifier
svm = SVC(random_state=42)

# Perform grid search
print("Running GridSearchCV (this may take a while)...")
grid_search = GridSearchCV(
    svm, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_train_hog_scaled, y_train)

# Get best parameters
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"\nBest parameters: {best_params}")
print(f"Best cross-validation score: {best_score:.4f}")

# Evaluate on test set
best_svm = grid_search.best_estimator_
y_pred_svm = best_svm.predict(X_test_hog_scaled)
test_accuracy = accuracy_score(y_test, y_pred_svm)

print(f"Test set accuracy: {test_accuracy:.4f}")
print(f"Improvement over baseline: {test_accuracy - results['HOG']['SVM']['accuracy']:.4f}")

## Cross-Validation Analysis

In [ ]:
# Perform cross-validation on top models
print("Performing cross-validation analysis...")

# Select top 3 models from baseline
top_models = results_df.nlargest(3, 'Accuracy')
print("\nTop 3 models:")
print(top_models[['Model', 'Feature Set', 'Accuracy']].to_string(index=False))

# Cross-validation results
cv_results = {}

for _, row in top_models.iterrows():
    model_name = row['Model']
    feature_set = row['Feature Set']
    
    print(f"\nCross-validating {model_name} with {feature_set} features...")
    
    # Get features
    X_tr, X_te = feature_sets[feature_set]
    
    # Scale if needed
    if model_name != 'Random Forest':
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
    
    # Create model
    if model_name == 'SVM':
        model = SVC(C=SVM_C, kernel=SVM_KERNEL, gamma=SVM_GAMMA, random_state=42)
    elif model_name == 'Random Forest':
        model = RandomForestClassifier(n_estimators=100, random_state=42)
    elif model_name == 'KNN':
        model = KNeighborsClassifier(n_neighbors=5)
    elif model_name == 'Logistic Regression':
        model = LogisticRegression(random_state=42, max_iter=1000)
    else:
        continue
    
    # Perform cross-validation
    scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='accuracy')
    
    cv_results[f"{model_name}_{feature_set}"] = {
        'scores': scores,
        'mean': scores.mean(),
        'std': scores.std(),
        'model': model_name,
        'feature_set': feature_set
    }
    
    print(f"  CV scores: {scores}")
    print(f"  Mean: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Plot cross-validation results
plt.figure(figsize=(12, 6))
model_names = list(cv_results.keys())
means = [cv_results[name]['mean'] for name in model_names]
stds = [cv_results[name]['std'] for name in model_names]

bars = plt.bar(model_names, means, yerr=stds, capsize=5, 
               color=['skyblue', 'lightgreen', 'lightcoral'])
plt.title('Cross-Validation Results (5-fold)')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=45)

# Add value labels on bars
for bar, mean in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{mean:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Feature Importance Analysis

In [ ]:
# Feature importance for Random Forest
print("Analyzing feature importance...")

# Train Random Forest on combined features
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_combined, y_train)

# Get feature importances
importances = rf_model.feature_importances_

# Create feature group labels
feature_groups = []
n_hog = X_train_hog.shape[1]
n_color = X_train_color.shape[1]
n_lbp = X_train_lbp.shape[1]

feature_groups.extend(['HOG'] * n_hog)
feature_groups.extend(['Color'] * n_color)
feature_groups.extend(['LBP'] * n_lbp)

# Calculate importance by feature group
group_importance = {}
for i, group in enumerate(feature_groups):
    if group not in group_importance:
        group_importance[group] = []
    group_importance[group].append(importances[i])

# Calculate mean importance per group
group_mean_importance = {group: np.mean(imp) for group, imp in group_importance.items()}

print("Feature group importance:")
for group, importance in group_mean_importance.items():
    print(f"  {group}: {importance:.4f}")

# Plot feature group importance
plt.figure(figsize=(10, 6))
groups = list(group_mean_importance.keys())
importances_plot = list(group_mean_importance.values())

bars = plt.bar(groups, importances_plot, color=['skyblue', 'lightcoral', 'lightgreen'])
plt.title('Feature Group Importance (Random Forest)')
plt.ylabel('Mean Importance')

# Add value labels on bars
for bar, imp in zip(bars, importances_plot):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{imp:.3f}', ha='center', va='bottom')

plt.show()

# Plot top individual features
top_indices = np.argsort(importances)[::-1][:20]
top_importances = importances[top_indices]
top_groups = [feature_groups[i] for i in top_indices]

plt.figure(figsize=(12, 6))
colors = {'HOG': 'skyblue', 'Color': 'lightcoral', 'LBP': 'lightgreen'}
bar_colors = [colors[group] for group in top_groups]

bars = plt.bar(range(20), top_importances, color=bar_colors)
plt.title('Top 20 Individual Features by Importance')
plt.xlabel('Feature Rank')
plt.ylabel('Importance')

# Create legend
legend_elements = [plt.Rectangle((0,0),1,1, facecolor=colors[group], label=group) 
                   for group in colors.keys()]
plt.legend(handles=legend_elements)

plt.show()

## Learning Curve Analysis

In [ ]:
# Learning curve analysis for best model
from sklearn.model_selection import learning_curve

print("Generating learning curves...")

# Use best model from previous analysis
best_model_name = top_models.iloc[0]['Model']
best_feature_set = top_models.iloc[0]['Feature Set']

X_tr, X_te = feature_sets[best_feature_set]

if best_model_name != 'Random Forest':
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)

# Create model
if best_model_name == 'SVM':
    model = SVC(C=SVM_C, kernel=SVM_KERNEL, gamma=SVM_GAMMA, random_state=42)
elif best_model_name == 'Random Forest':
    model = RandomForestClassifier(n_estimators=100, random_state=42)
elif best_model_name == 'KNN':
    model = KNeighborsClassifier(n_neighbors=5)
else:
    model = LogisticRegression(random_state=42, max_iter=1000)

# Generate learning curve
train_sizes, train_scores, val_scores = learning_curve(
    model, X_tr, y_train, cv=5, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    random_state=42
)

# Calculate means and standard deviations
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Plot learning curve
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', color='red', label='Validation Score')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')

plt.title(f'Learning Curve - {best_model_name} ({best_feature_set})')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.show()

# Analyze overfitting/underfitting
final_train_score = train_mean[-1]
final_val_score = val_mean[-1]
gap = final_train_score - final_val_score

print(f"Learning curve analysis:")
print(f"  Final training score: {final_train_score:.4f}")
print(f"  Final validation score: {final_val_score:.4f}")
print(f"  Gap (overfitting indicator): {gap:.4f}")

if gap > 0.1:
    print("  ⚠️  Possible overfitting - consider regularization")
elif gap < 0.05:
    print("  ✅ Good generalization")
else:
    print("  ℹ️  Moderate overfitting")

## Error Analysis

In [ ]:
# Analyze misclassified examples
print("Performing error analysis...")

# Use best model
X_tr, X_te = feature_sets[best_feature_set]

if best_model_name != 'Random Forest':
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

# Train model
if best_model_name == 'SVM':
    model = SVC(C=SVM_C, kernel=SVM_KERNEL, gamma=SVM_GAMMA, random_state=42)
elif best_model_name == 'Random Forest':
    model = RandomForestClassifier(n_estimators=100, random_state=42)
else:
    model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_tr, y_train)
y_pred = model.predict(X_te)

# Find misclassified examples
misclassified_indices = np.where(y_pred != y_test)[0]
print(f"Number of misclassified examples: {len(misclassified_indices)}")
print(f"Misclassification rate: {len(misclassified_indices)/len(y_test):.4f}")

# Analyze confusion patterns
cm = confusion_matrix(y_test, y_pred)
classes = label_encoder.classes_

print("\nConfusion patterns:")
for i in range(len(classes)):
    for j in range(len(classes)):
        if i != j and cm[i, j] > 0:
            print(f"  {classes[i]} → {classes[j]}: {cm[i, j]} cases")

# Visualize some misclassified examples
if len(misclassified_indices) > 0:
    n_examples = min(6, len(misclassified_indices))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, idx in enumerate(misclassified_indices[:n_examples]):
        # Get image
        img = X_test[idx]
        img_display = (img * 255).astype(np.uint8)
        
        # Convert to RGB for display
        if len(img_display.shape) == 3:
            img_display = cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)
        
        # Display image
        axes[i].imshow(img_display)
        true_label = label_encoder.inverse_transform([y_test[idx]])[0]
        pred_label = label_encoder.inverse_transform([y_pred[idx]])[0]
        axes[i].set_title(f'True: {true_label}\nPred: {pred_label}')
        axes[i].axis('off')
    
    # Hide unused subplots
    for i in range(n_examples, 6):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples to display!")

## Summary and Recommendations

In [ ]:
# Generate final summary
print("Model Experiments Summary:")
print("=" * 50)

# Best overall model
best_overall = results_df.loc[results_df['Accuracy'].idxmax()]
print(f"\n🏆 Best Overall Model:")
print(f"   Model: {best_overall['Model']}")
print(f"   Features: {best_overall['Feature Set']}")
print(f"   Accuracy: {best_overall['Accuracy']:.4f}")
print(f"   F1-Score: {best_overall['F1-Score']:.4f}")

# Feature set comparison
feature_comparison = results_df.groupby('Feature Set')['Accuracy'].max()
print(f"\n📊 Best Feature Sets:")
for feature_set, accuracy in feature_comparison.items():
    print(f"   {feature_set}: {accuracy:.4f}")

# Model comparison
model_comparison = results_df.groupby('Model')['Accuracy'].max()
print(f"\n🤖 Best Models:")
for model, accuracy in model_comparison.items():
    print(f"   {model}: {accuracy:.4f}")

print(f"\n💡 Recommendations:")
print(f"1. Use {best_overall['Model']} with {best_overall['Feature Set']} features for production")
print(f"2. Consider ensemble methods for potentially better performance")
print(f"3. Feature selection could improve efficiency without sacrificing accuracy")
print(f"4. Monitor for overfitting in production")
print(f"5. Consider data augmentation if more data is needed")

# Save best model
print(f"\n💾 Saving best model...")
import pickle
from config import MODEL_DIR

os.makedirs(MODEL_DIR, exist_ok=True)

# Save model
model_path = os.path.join(MODEL_DIR, 'best_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# Save label encoder
encoder_path = os.path.join(MODEL_DIR, 'best_label_encoder.pkl')
with open(encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)

# Save scaler if used
if best_model_name != 'Random Forest':
    scaler_path = os.path.join(MODEL_DIR, 'best_scaler.pkl')
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)

print(f"Model saved to: {model_path}")
print(f"Label encoder saved to: {encoder_path}")
if best_model_name != 'Random Forest':
    print(f"Scaler saved to: {scaler_path}")